<a href="https://colab.research.google.com/github/kunjana30/Internship-Project/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***IBI*** ***Project***

*importing* *libraries*

In [1]:
import pandas as pd
import numpy as np

*setting  random seed for producability*

In [2]:
np.random.seed(42)

*defining configuration*

In [3]:
dates = pd.date_range(start="2025-01-01",end="2025-12-31", freq="D")
zones = [f"Zone_{str(i).zfill(2)}" for i in range(1,6)]
data_rows = []

*simulate data for ech zone an date*

In [4]:
for zone in zones:
  for date in dates:

    #Base temp modeling seasonal variations
    day_of_year = date.dayofyear
    base_temp = 20 + 10 * np.sin(2 * np.pi * (day_of_year-150) / 365)
    avg_temp = round(base_temp + np.random.normal(0,3), 1)

    #Humidity is inversely proportional to temperature with noise
    base_humidity = 80 - (avg_temp * 1.2)
    humidity = int(np.clip(base_humidity + np.random.normal(0,10), 10, 100))

    #5% chance of an event
    special_event = 1 if np.random.rand() < 0.05 else 0

    #Simulating energy consumption
    base_load = 5000
    cooling_load = max(0, (avg_temp - 22) * 250) if avg_temp > 22 else 0
    heating_load = max(0, (15 - avg_temp) * 150) if avg_temp < 15 else 0
    event_load = special_event * np.random.uniform(1500,3000)
    noise = np.random.normal(0,400)

    energy_consumption = round(base_load + cooling_load + heating_load + event_load + noise, 2)

    data_rows.append({
        "Date": date.strftime("%Y-%m-%d"),
        "ZoneID": zone,
        "AvgTemperature": avg_temp,
        "Humidity": humidity,
        "SpecialEvent": special_event,
        "EnergyConsumption(kWh)": energy_consumption
    })

*creating data frame and export*

In [5]:
df = pd.DataFrame(data_rows)
df.to_csv("synthetic_energy_dataset.csv", index = False)

*preview the data*



In [6]:
print(df.head(5))

         Date   ZoneID  AvgTemperature  Humidity  SpecialEvent  \
0  2025-01-01  Zone_01            16.0        59             0   
1  2025-01-02  Zone_01            13.7        79             1   
2  2025-01-03  Zone_01            12.9        69             0   
3  2025-01-04  Zone_01             8.4        52             0   
4  2025-01-05  Zone_01            11.3        52             0   

   EnergyConsumption(kWh)  
0                 4906.34  
1                 8456.84  
2                 5411.78  
3                 5765.08  
4                 5582.01  


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [8]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by = ['ZoneID', 'Date'])

*creating next day's energy demand*

In [9]:
df['NextDayDemand'] = df.groupby('ZoneID')['EnergyConsumption(kWh)'].shift(-1)
df = df.dropna()

*selecting features(x) and targets(y)*

In [10]:
features = ['AvgTemperature', 'Humidity', 'SpecialEvent', 'EnergyConsumption(kWh)']
x = df[features]
y = df['NextDayDemand']

*splitting into training and testing sets*

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

*scaling the features*

In [12]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

*training a model*

In [13]:
model = RandomForestRegressor(random_state=42)
model.fit(x_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [14]:
predictions = model.predict(x_test_scaled)
mae = mean_absolute_error(y_test, predictions)
print(f"Model Training Complete!")
print(f"Mean Absolute Error: {mae:.2f} kWh")

Model Training Complete!
Mean Absolute Error: 710.10 kWh


In [15]:
pip install streamlit pandas numpy plotly scikit-learn statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 98.5 MB/s eta 0:00:00


In [16]:
%%writefile app.py
import pandas as pd
import numpy as np
import streamlit as st
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px

st.set_page_config(page_title="City Energy Dashboard", layout="wide", page_icon="⚡")

@st.cache_data
def load_data():
    np.random.seed(42)
    dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")
    zones = [f"Zone_{str(i).zfill(2)}" for i in range(1, 6)]

    data = []
    for zone in zones:
        for date in dates:
            day_of_year = date.dayofyear
            base_temp = 20 + 10 * np.sin(2 * np.pi * (day_of_year - 150) / 365)
            avg_temp = round(base_temp + np.random.normal(0, 3), 1)

            base_humidity = 80 - (avg_temp * 1.2)
            humidity = int(np.clip(base_humidity + np.random.normal(0, 10), 10, 100))

            special_event = 1 if np.random.rand() < 0.05 else 0

            base_load_val = 5000
            cooling_load = max(0, (avg_temp - 22) * 250) if avg_temp > 22 else 0
            heating_load = max(0, (15 - avg_temp) * 150) if avg_temp < 15 else 0
            event_load = special_event * np.random.uniform(1500, 3000)
            noise = np.random.normal(0, 400)

            energy_consumption = round(base_load_val + cooling_load + heating_load + event_load + noise, 2)

            data.append([
                date,
                zone,
                avg_temp,
                humidity,
                special_event,
                energy_consumption
            ])

    df = pd.DataFrame(data, columns=['Date', 'ZoneID', 'AvgTemperature', 'Humidity', 'SpecialEvent', 'EnergyConsumption(kWh)'])
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['ZoneID', 'Date'])
    df['NextDayDemand'] = df.groupby('ZoneID')['EnergyConsumption(kWh)'].shift(-1)
    return df.dropna()

df = load_data()

@st.cache_resource
def train_model(data):
    features = ['ZoneID', 'AvgTemperature', 'Humidity', 'SpecialEvent', 'EnergyConsumption(kWh)']
    x = data[features]
    y = data['NextDayDemand']

    # Preprocessing: OneHotEncode ZoneID + Scale numerical features
    categorical_features = ['ZoneID']
    numeric_features = ['AvgTemperature', 'Humidity', 'SpecialEvent', 'EnergyConsumption(kWh)']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', StandardScaler(), numeric_features)
        ]
    )

    model = RandomForestRegressor(random_state=42, n_estimators=200, max_depth=12)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])
    pipeline.fit(x, y)

    return pipeline, features

model_pipeline, features = train_model(df)

st.title("City Energy Consumption Analysis and Prediction")
st.markdown("Monitor historical usage patterns and predict next-day energy grid demand")

st.sidebar.header("Next-Day Demand Predictor")
selected_zone = st.sidebar.selectbox("Select target zone", df['ZoneID'].unique())
input_temp = st.sidebar.slider("Forecasted Temperature (℃)", 10.0, 45.0, 25.0)
input_humidity = st.sidebar.slider("Forecasted Humidity (%)", 0, 100, 50)
input_event = st.sidebar.checkbox("Special Event / Holiday Tomorrow")
input_current_load = st.sidebar.number_input("Today's Total Consumption (kWh)", min_value=1000, max_value=20000, value=6000)

if st.sidebar.button("Predict Next-Day Demand"):
    event_binary = 1 if input_event else 0
    user_features = pd.DataFrame([[selected_zone, input_temp, input_humidity, event_binary, input_current_load]],
                                 columns=features)
    predicted_demand = model_pipeline.predict(user_features)[0]
    st.sidebar.success(f"### Predicted Demand for {selected_zone}:\n## {predicted_demand:,.2f} kWh")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Historical Energy Consumption")
    zone_filter = st.multiselect("Filter Zones", options=df['ZoneID'].unique(), default=df['ZoneID'].unique())
    filtered_df = df[df['ZoneID'].isin(zone_filter)]
    fig_line = px.line(filtered_df, x='Date', y='EnergyConsumption(kWh)', color='ZoneID',
                       title="Daily Load Profiling", labels={'EnergyConsumption(kWh)': 'Demand (kWh)'})
    st.plotly_chart(fig_line, use_container_width=True)

with col2:
    st.subheader("Temperature v/s Energy Load")
    fig_scatter = px.scatter(df, x='AvgTemperature', y='EnergyConsumption(kWh)', color='ZoneID',
                             trendline="ols", title="Weather Impact Analysis")
    st.plotly_chart(fig_scatter, use_container_width=True)

st.subheader("Zone Demand Distribution Summary")
fig_box = px.box(df, x='ZoneID', y='EnergyConsumption(kWh)', color='ZoneID', title="Spread of Consumption per zone")
st.plotly_chart(fig_box, use_container_width=True)

st.subheader("Monthly Demand Heatmap")
df['Month'] = df['Date'].dt.month
heatmap_data = df.groupby(['ZoneID', 'Month'])['EnergyConsumption(kWh)'].mean().reset_index()
fig_heatmap = px.density_heatmap(heatmap_data, x='Month', y='ZoneID', z='EnergyConsumption(kWh)',
                                 color_continuous_scale='Viridis', title="Average Monthly Demand by Zone")
st.plotly_chart(fig_heatmap, use_container_width=True)

st.subheader("Feature Importance (Random Forest)")
importances = model_pipeline.named_steps['model'].feature_importances_
feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out(features)
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)
fig_importance = px.bar(importance_df, x='Importance', y='Feature', orientation='h', title="Feature Importance")
st.plotly_chart(fig_importance, use_container_width=True)


Writing app.py


In [17]:
!pip install streamlit plotly scikit-learn
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 5s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [18]:
# Start Streamlit in the background, redirecting output to a log file
!nohup streamlit run app.py > streamlit_output.log 2>&1 &

# Wait a moment for Streamlit to fully start
!sleep 5

# Start localtunnel in the background and redirect its stdout and stderr to a log file.
!nohup npx localtunnel --port 8501 > localtunnel_output.log 2>&1 &

# Give localtunnel some time to establish the tunnel and write the URL to the log
!sleep 10 # Increased sleep time for tunnel establishment

# Now, keep the cell alive by continuously monitoring the localtunnel_output.log file
print("Your Streamlit app should be accessible at the URL printed below. Keep this cell running!")
print("Waiting for localtunnel URL...")

import time
# Loop until URL is found, or for a reasonable timeout (e.g., 60 seconds)
for _ in range(20): # Try for 20 * 3 seconds = 60 seconds
    with open('localtunnel_output.log', 'r') as f:
        content = f.read()
        if "your url is:" in content:
            # Extract URL from the log file. Assuming the URL is on a line with 'your url is:'
            for line in content.split('\n'):
                if "your url is:" in line:
                    url = line.split("your url is:")[1].strip()
                    print(f"Public URL: {url}")
                    break
            break
    time.sleep(3)
else:
    print("Could not retrieve localtunnel URL within timeout. Check localtunnel_output.log for errors.")

# Keep the cell alive and show any further localtunnel output (e.g., connection status)
!tail -f localtunnel_output.log

Your Streamlit app should be accessible at the URL printed below. Keep this cell running!
Waiting for localtunnel URL...
Public URL: https://tricky-women-see.loca.lt
your url is: https://tricky-women-see.loca.lt
^C
